In [19]:
# Content-Based Filtering
# Objetivo: recomendar películas similares basándonos en géneros

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Cargamos los datos limpios
movies = pd.read_csv('../data/movies_clean.csv')

print(movies.shape)
movies.head()

(2269, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,5,Father of the Bride Part II (1995),Comedy
4,6,Heat (1995),Action|Crime|Thriller


In [20]:
"""
Para esto vamos a usar una técnica llamada TF-IDF + Similitud del Coseno 
— que básicamente convierte los géneros de cada película en vectores numéricos y mide qué tan parecidos son entre sí.
"""
#Preparar los géneros

# Reemplazamos "|" por espacios para que TF-IDF trate cada género como una palabra
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)
movies[['title', 'genres', 'genres_clean']].head()


,title,genres,genres_clean
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,Father of the Bride Part II (1995),Comedy,Comedy
4,Heat (1995),Action|Crime|Thriller,Action Crime Thriller


In [21]:
#Construir la matriz TF-IDF

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])

print("Dimensiones de la matriz:", tfidf_matrix.shape)
print("Géneros detectados:", tfidf.get_feature_names_out())

Dimensiones de la matriz: (2269, 21)
Géneros detectados: ['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'fi' 'film' 'horror' 'imax' 'musical'
 'mystery' 'noir' 'romance' 'sci' 'thriller' 'war' 'western']


In [22]:
# Calcular similitud entre películas

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print("Dimensiones de la matriz de similitud:", cosine_sim.shape)

Dimensiones de la matriz de similitud: (2269, 2269)


In [23]:
"""
Interpretaciones
Matriz TF-IDF (2269, 21) → 2.269 películas representadas con 21 géneros únicos como columnas
Matriz de similitud (2269, 2269) → cada película comparada contra todas las demás

Hay un detalle a corregir y es que Sci-Fi se dividió en 'sci' y 'fi' porque TF-IDF separó por el guión. 
"""

"\nInterpretaciones\nMatriz TF-IDF (2269, 21) → 2.269 películas representadas con 21 géneros únicos como columnas\nMatriz de similitud (2269, 2269) → cada película comparada contra todas las demás\n\nHay un detalle a corregir y es que Sci-Fi se dividió en 'sci' y 'fi' porque TF-IDF separó por el guión. \n"

In [24]:
movies['genres_clean'] = movies['genres_clean'].str.replace('Sci-Fi', 'SciFi', regex=False)
movies['genres_clean'] = movies['genres_clean'].str.replace('Film-Noir', 'FilmNoir', regex=False)

# Reconstruimos la matriz con la corrección
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])

print("Géneros detectados:", tfidf.get_feature_names_out())

Géneros detectados: ['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'filmnoir' 'horror' 'imax' 'musical'
 'mystery' 'romance' 'scifi' 'thriller' 'war' 'western']


In [25]:
#Recalcular similitud entre 

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print("Dimensiones:", cosine_sim.shape)

Dimensiones: (2269, 2269)


In [26]:
# construir la función que dado el nombre de una película, devuelva las más similares.
# Creamos un índice para buscar películas por título
indices = pd.Series(movies.index, index=movies['title'])
print(indices.head(10))


title
Toy Story (1995)                      0
Jumanji (1995)                        1
Grumpier Old Men (1995)               2
Father of the Bride Part II (1995)    3
Heat (1995)                           4
Sabrina (1995)                        5
Sudden Death (1995)                   6
GoldenEye (1995)                      7
American President, The (1995)        8
Dracula: Dead and Loving It (1995)    9
dtype: int64


In [27]:
#Funcion de recomendacion

def get_recommendations(title, n=10):
    # Obtenemos el índice de la película
    idx = indices[title]
    
    # Obtenemos los scores de similitud con todas las demás
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Ordenamos por similitud descendente
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Tomamos las n más similares (excluimos la primera que es ella misma)
    sim_scores = sim_scores[1:n+1]
    
    # Obtenemos los índices de las películas
    movie_indices = [i[0] for i in sim_scores]
    
    return movies[['title', 'genres']].iloc[movie_indices]

In [28]:
#se prueba el modelo
get_recommendations("Toy Story (1995)")

,title,genres
847,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
1104,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
1315,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
1421,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
1875,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy
2259,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy
2236,Inside Out (2015),Adventure|Animation|Children|Comedy|Drama|Fantasy
800,"Lord of the Rings, The (1978)",Adventure|Animation|Children|Fantasy
1374,Atlantis: The Lost Empire (2001),Adventure|Animation|Children|Fantasy
1978,Ponyo (Gake no ue no Ponyo) (2008),Adventure|Animation|Children|Fantasy


In [29]:
"""
Si el usuario buscaría get_recommendations("toy story") por ejemplo tendríamos un error de index por lo cual hay que preveer estas.
"""

'\nSi el usuario buscaría get_recommendations("toy story") por ejemplo tendríamos un error de index por lo cual hay que preveer estas.\n'

In [30]:
def get_recommendations_part(title, n=10):
    # Paso 1 - Búsqueda parcial e insensible a mayúsculas
    matches = indices[indices.index.str.lower().str.contains(title.lower())]
    
    if len(matches) == 0:
        return "No se encontró ninguna película con ese título"
    
    # Paso 2 - Tomamos el primer resultado
    """
    matches es una Serie de pandas donde:
    El índice son los títulos → matches.index[0]
    Los valores son los índices numéricos → matches.iloc[0]
    """
    idx = matches.iloc[0]  # ¿cómo obtenés el primer valor de matches?
    
    # Paso 3 - Similitud (esto ya lo tenías bien)
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]
    
    movie_indices = [i[0] for i in sim_scores]
    
    # Paso 4 - ¿Qué película encontramos realmente?
    print(f"Mostrando recomendaciones para: {matches.index[0]}")
    
    return movies[['title', 'genres']].iloc[movie_indices]

In [31]:
get_recommendations_part("toy story")
get_recommendations_part("Toy Story")
get_recommendations_part("toy")

Mostrando recomendaciones para: Toy Story (1995)
Mostrando recomendaciones para: Toy Story (1995)
Mostrando recomendaciones para: Toy Story (1995)


,title,genres
847,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
1104,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
1315,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
1421,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
1875,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy
2259,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy
2236,Inside Out (2015),Adventure|Animation|Children|Comedy|Drama|Fantasy
800,"Lord of the Rings, The (1978)",Adventure|Animation|Children|Fantasy
1374,Atlantis: The Lost Empire (2001),Adventure|Animation|Children|Fantasy
1978,Ponyo (Gake no ue no Ponyo) (2008),Adventure|Animation|Children|Fantasy
